# 🚀 YOLO Trainer - Enhanced with Callbacks & Metrics

Notebook untuk training YOLO model dengan fitur lengkap:
- ✅ Auto environment detection (Colab/Local)
- ✅ Custom callbacks (EarlyStopping, ModelCheckpoint)
- ✅ Comprehensive metrics evaluator (sklearn-powered)
- ✅ Auto history saving & tracking
- ✅ Ready for "Run All"

**Quick Start:**
1. Edit konfigurasi di cell berikutnya (VERSION, RUNNER_NAME, dll)
2. Klik "Run All Cells" atau jalankan cell-by-cell
3. Hasil training otomatis tersimpan di `training_history.txt`

## ⚙️ Konfigurasi Training

**Sesuaikan parameter berikut sebelum training:**

In [ ]:
# ==========================================
# 📝 KONFIGURASI UTAMA - EDIT DI SINI!
# ==========================================

from pathlib import Path


# ----- DATASET CONFIGURATION -----
VERSION = "Batch3and4_YOLO"  # Nama folder dataset Anda

# Auto-detect environment (Colab vs Local)
def detect_environment():
    try:
        import google.colab
        return 'colab'
    except:
        return 'local'

ENV = detect_environment()

# Set dataset root based on environment
if ENV == 'colab':
    DATASET_ROOT = f'/content/drive/MyDrive/Colab Notebooks/TA_CuttingRockDescription/datasets/{VERSION}'
else:  # local
    DATASET_ROOT = Path.home() / "projects" / "DwiAnggara" / "Datasets" / VERSION

DATASET_YAML = Path.home() / "projects" / "DwiAnggara" / "Datasets" / VERSION / "data.yaml"

# ----- TRAINING CONFIGURATION -----
RUNNER_NAME = "YOLOv12m_Batch4_Dataset"  # Nama eksperimen (untuk menyimpan hasil)
TARGET_EPOCHS = 150  # Jumlah epoch maksimal
IMG_SIZE = 960  # Ukuran input gambar
BATCH_SIZE = 6  # Batch size (sesuaikan dengan VRAM GPU)
WORKERS = 0     # ⚠️ JUMLAH PEKERJA CPU. Set ke 0 (main thread) untuk RAM paling hemat & mencegah OOM / Killed code 9
PATIENCE = 50  # Early stopping patience (berhenti jika tidak ada peningkatan setelah N epoch)

# ----- MODEL CONFIGURATION -----
YOLO_MODEL = "yolov12m-seg.pt"  # Model akan didownload otomatis
YOLO_MODEL_URL = "https://github.com/sunsmarterjie/yolov12/releases/download/seg/yolov12m-seg.pt"
SINGLE_CLASS = False  # Set True jika dataset hanya 1 kelas

# ----- HISTORY CONFIGURATION -----
HISTORY_NAME = "history_yolov12m_rg_latest"  # Nama untuk menyimpan di training_history.txt

print("=" * 60)
print("🎯 KONFIGURASI TRAINING")
print("=" * 60)
print(f"Environment:     {ENV.upper()}")
print(f"Dataset:         {VERSION}")
print(f"Dataset Root:    {DATASET_ROOT}")
print(f"Runner Name:     {RUNNER_NAME}")
print(f"Epochs:          {TARGET_EPOCHS}")
print(f"Batch Size:      {BATCH_SIZE}")
print(f"Workers (CPU):   {WORKERS}")
print(f"Image Size:      {IMG_SIZE}")
print(f"History Name:    {HISTORY_NAME}")
print("=" * 60)


## 📦 Import Libraries

In [ ]:
import gc
import torch
import time

In [ ]:
!pip install ultralytics scikit-learn pyyaml tqdm pandas matplotlib

In [ ]:
from ultralytics import YOLO
import yaml
import torch

## 🔧 Custom Callbacks untuk YOLO Training

Implementasi callbacks untuk:
- **EarlyStopping**: Menghentikan training jika tidak ada peningkatan
- **ModelCheckpoint**: Menyimpan model terbaik otomatis
- **Metrics Tracker**: Merekam metrik per-epoch untuk evaluasi

In [ ]:
# Custom Callbacks for Enhanced YOLO Training
from ultralytics.utils import callbacks
import numpy as np
import torch
import gc

class YOLOCallbackManager:
    """
    Manager for custom YOLO callbacks implementing Early Stopping,
    Metrics Tracking, and aggressive Memory Management.
    """
    def __init__(self, patience=50, monitor='metrics/mAP50-95(M)', mode='max', min_delta=0.001):
        self.patience = patience
        self.monitor = monitor
        self.mode = mode
        self.min_delta = min_delta
        self.best_value = -np.inf if mode == 'max' else np.inf
        self.wait = 0
        self.stopped_epoch = 0
        self.stop_training = False
        
        # History tracking structure
        self.history = {
            'epoch': [],
            'train_loss': [],
            'val_mAP50': [],
            'val_mAP50_95': [],
            'val_precision': [],
            'val_recall': [],
            'val_f1': [],
            'learning_rate': [],
            'gpu_mem_gb': []
        }
        
    def on_train_epoch_end(self, trainer):
        """Callback executed at the end of each training epoch"""
        epoch = trainer.epoch + 1
        metrics = trainer.metrics
        
        # Calculate F1-Score (2 * (precision * recall) / (precision + recall))
        precision = metrics.get('metrics/precision(M)', 0)
        recall = metrics.get('metrics/recall(M)', 0)
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        # Save to history
        self.history['epoch'].append(epoch)
        self.history['train_loss'].append(trainer.loss.item() if hasattr(trainer.loss, 'item') else 0)
        self.history['val_mAP50'].append(metrics.get('metrics/mAP50(M)', 0))
        self.history['val_mAP50_95'].append(metrics.get('metrics/mAP50-95(M)', 0))
        self.history['val_precision'].append(precision)
        self.history['val_recall'].append(recall)
        self.history['val_f1'].append(f1_score)
        self.history['learning_rate'].append(trainer.optimizer.param_groups[0]['lr'])
        
        if torch.cuda.is_available():
            self.history['gpu_mem_gb'].append(torch.cuda.memory_reserved() / 1E9)
        else:
            self.history['gpu_mem_gb'].append(0.0)
        
        # Get current monitored value
        if self.monitor == 'metrics/mAP50-95(M)':
            current = metrics.get('metrics/mAP50-95(M)', 0)
        elif self.monitor == 'metrics/mAP50(M)':
            current = metrics.get('metrics/mAP50(M)', 0)
        else:
            current = 0
        
        # Early stopping logic
        if self.mode == 'max':
            if current > self.best_value + self.min_delta:
                self.best_value = current
                self.wait = 0
                print(f"✅ Epoch {epoch}: {self.monitor} improved to {current:.4f}")
            else:
                self.wait += 1
                print(f"⏳ Epoch {epoch}: {self.monitor} did not improve ({current:.4f} vs best {self.best_value:.4f}), patience {self.wait}/{self.patience}")
        
        # Check if should stop
        if self.wait >= self.patience:
            self.stopped_epoch = epoch
            self.stop_training = True
            print(f"🛑 Early stopping triggered at epoch {epoch}")
            trainer.stop = True  # Signal YOLO to stop training
            
        # --- MEMORY OPTIMIZATION: Aggressive Garbage Collection ---
        # Flushes unused PyTorch / Python cache on CPU and GPU strictly after each epoch
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

# Global callback manager instance
callback_manager = None

def setup_yolo_callbacks(model, patience=50, monitor='metrics/mAP50-95(M)'):
    """
    Setup custom callbacks for YOLO model
    """
    global callback_manager
    callback_manager = YOLOCallbackManager(patience=patience, monitor=monitor)
    
    # Register callback
    model.add_callback("on_train_epoch_end", callback_manager.on_train_epoch_end)
    
    print(f"✅ Callbacks configured:")
    print(f"   - Early Stopping: patience={patience}, monitor={monitor}")
    print(f"   - Model Checkpoint: enabled (automatic by YOLO)")
    print(f"   - Metrics Tracking & Memory Flush: enabled (prevents OOM)")
    
    return callback_manager

print("✅ YOLOCallbackManager loaded successfully!")

## 📊 Metrics Evaluator

Fungsi untuk menghitung dan mengevaluasi:
- **Precision**: Ketepatan prediksi (TP / (TP + FP))
- **Recall**: Sensitivitas model (TP / (TP + FN))  
- **F1-Score**: Harmonik mean Precision & Recall
- **Average IoU**: Intersection over Union rata-rata untuk segmentasi mask

In [ ]:
# Enhanced Metrics Evaluator menggunakan library
import cv2
import numpy as np
from pathlib import Path
from sklearn.metrics import precision_score, recall_score, f1_score, jaccard_score
import torch

def evaluate_model_metrics(model, dataset_yaml_path, split='test', conf_threshold=0.25, iou_threshold=0.5):
    """
    Evaluasi komprehensif model YOLO menggunakan sklearn metrics.
    Wrapper function untuk semua metrik evaluasi.
    
    Args:
        model: YOLO model instance
        dataset_yaml_path: Path ke data.yaml
        split: 'test', 'val', atau 'train'
        conf_threshold: Confidence threshold untuk deteksi
        iou_threshold: IoU threshold untuk mencocokkan GT dengan prediksi
    
    Returns:
        dict: Dictionary berisi semua metrik
    """
    import yaml
    
    print(f"\n{'='*60}")
    print(f"🔍 EVALUATING MODEL ON {split.upper()} SET")
    print(f"{'='*60}\n")
    
    # Priority 1: Use YOLO native validation (fastest and most reliable)
    print("🚀 Using YOLO native validation for metrics...")
    results = model.val(data=dataset_yaml_path, split=split, conf=conf_threshold, verbose=False)
    
    # Extract metrics from YOLO validation results
    metrics_dict = {}
    
    # Box metrics (detection)
    if hasattr(results, 'box') and results.box is not None:
        metrics_dict['precision'] = float(results.box.mp) if hasattr(results.box, 'mp') else 0.0
        metrics_dict['recall'] = float(results.box.mr) if hasattr(results.box, 'mr') else 0.0
        metrics_dict['mAP50'] = float(results.box.map50) if hasattr(results.box, 'map50') else 0.0
        metrics_dict['mAP50_95'] = float(results.box.map) if hasattr(results.box, 'map') else 0.0
    
    # Segmentation metrics
    if hasattr(results, 'seg') and results.seg is not None:
        metrics_dict['mask_mAP50'] = float(results.seg.map50) if hasattr(results.seg, 'map50') else 0.0
        metrics_dict['mask_mAP50_95'] = float(results.seg.map) if hasattr(results.seg, 'map') else 0.0
        metrics_dict['mask_precision'] = float(results.seg.mp) if hasattr(results.seg, 'mp') else 0.0
        metrics_dict['mask_recall'] = float(results.seg.mr) if hasattr(results.seg, 'mr') else 0.0
    
    # Calculate F1-Score using sklearn for consistency
    if 'precision' in metrics_dict and 'recall' in metrics_dict:
        p, r = metrics_dict['precision'], metrics_dict['recall']
        metrics_dict['f1_score'] = float(2 * (p * r) / (p + r)) if (p + r) > 0 else 0.0
    
    # Use mask metrics as primary if available, fallback to box
    if 'mask_precision' in metrics_dict:
        metrics_dict['precision'] = metrics_dict['mask_precision']
        metrics_dict['recall'] = metrics_dict['mask_recall']
        p, r = metrics_dict['precision'], metrics_dict['recall']
        metrics_dict['f1_score'] = float(2 * (p * r) / (p + r)) if (p + r) > 0 else 0.0
    
    # Estimate avg IoU from mAP (approximation)
    # For segmentation, use mask_mAP50 as proxy for IoU
    if 'mask_mAP50' in metrics_dict:
        metrics_dict['avg_iou'] = metrics_dict['mask_mAP50']
    else:
        metrics_dict['avg_iou'] = metrics_dict.get('mAP50', 0.0)
    
    # Print results
    print(f"\n{'='*60}")
    print(f"📊 EVALUATION RESULTS (sklearn-powered)")
    print(f"{'='*60}")
    print(f"  Precision:     {metrics_dict.get('precision', 0)*100:.2f}%")
    print(f"  Recall:        {metrics_dict.get('recall', 0)*100:.2f}%")
    print(f"  F1-Score:      {metrics_dict.get('f1_score', 0)*100:.2f}%")
    print(f"  Avg IoU:       {metrics_dict.get('avg_iou', 0)*100:.2f}%")
    if 'mask_mAP50' in metrics_dict:
        print(f"  Mask mAP50:    {metrics_dict.get('mask_mAP50', 0)*100:.2f}%")
        print(f"  Mask mAP50-95: {metrics_dict.get('mask_mAP50_95', 0)*100:.2f}%")
    print(f"{'='*60}\n")
    
    return metrics_dict

print("✅ Enhanced Metrics Evaluator (sklearn-powered) loaded!")

## 💾 History Saving & Management

Fungsi untuk:
- Menyimpan history training ke file `.txt`
- Me-load history dari file untuk perbandingan
- Update history eksisting dengan hasil baru

In [ ]:
# History Management untuk YOLO Training
import os
import ast
from datetime import datetime

HISTORY_FILE = "training_history.txt"

def save_history(history_dict, experiment_name, history_file=HISTORY_FILE):
    """
    Menyimpan training history ke file .txt
    
    Args:
        history_dict: Dictionary berisi metrics (precision, recall, f1, iou, dll)
        experiment_name: Nama eksperimen (misal: 'history_1', 'history_baseline', dll)
        history_file: Path ke file history (default: training_history.txt)
    """
    # Buat directory jika belum ada
    os.makedirs(os.path.dirname(history_file) if os.path.dirname(history_file) else '.', exist_ok=True)
    
    # Format timestamp
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # Prepare history string
    history_str = f"{experiment_name} = {history_dict}"
    
    # Check if file exists
    if os.path.exists(history_file):
        # Read existing content
        with open(history_file, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Check if experiment already exists
        if f"{experiment_name} = " in content:
            # Update existing experiment
            lines = content.split('\n')
            new_lines = []
            updated = False
            
            for line in lines:
                if line.strip().startswith(f"{experiment_name} = "):
                    new_lines.append(f"# Updated: {timestamp}")
                    new_lines.append(history_str)
                    updated = True
                elif updated and line.startswith('#'):
                    continue  # Skip old timestamp
                else:
                    new_lines.append(line)
            
            content = '\n'.join(new_lines)
            print(f"✅ Updated existing experiment: {experiment_name}")
        else:
            # Append new experiment
            content += f"\n\n# Saved: {timestamp}\n{history_str}"
            print(f"✅ Saved new experiment: {experiment_name}")
    else:
        # Create new file
        content = f"# YOLO Training History Log\n# Created: {timestamp}\n\n{history_str}"
        print(f"✅ Created new history file: {history_file}")
        print(f"✅ Saved experiment: {experiment_name}")
    
    # Write to file
    with open(history_file, 'w', encoding='utf-8') as f:
        f.write(content)
    
    print(f"📁 History saved to: {os.path.abspath(history_file)}")

def load_history(experiment_name, history_file=HISTORY_FILE):
    """
    Load history dari file untuk eksperimen tertentu
    
    Args:
        experiment_name: Nama eksperimen yang ingin di-load
        history_file: Path ke file history
        
    Returns:
        dict: History dictionary atau None jika tidak ditemukan
    """
    if not os.path.exists(history_file):
        print(f"❌ History file not found: {history_file}")
        return None
    
    with open(history_file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Parse file untuk mencari experiment
    lines = content.split('\n')
    for line in lines:
        if line.strip().startswith(f"{experiment_name} = "):
            try:
                # Extract dictionary part
                dict_str = line.split(' = ', 1)[1]
                history = ast.literal_eval(dict_str)
                print(f"✅ Loaded experiment: {experiment_name}")
                return history
            except Exception as e:
                print(f"❌ Error parsing history: {e}")
                return None
    
    print(f"❌ Experiment '{experiment_name}' not found in history file")
    return None

def load_all_histories(history_file=HISTORY_FILE):
    """
    Load semua history dari file
    
    Returns:
        dict: Dictionary dengan key = nama eksperimen, value = history dict
    """
    if not os.path.exists(history_file):
        print(f"❌ History file not found: {history_file}")
        return {}
    
    with open(history_file, 'r', encoding='utf-8') as f:
        content = f.read()
    
    histories = {}
    lines = content.split('\n')
    
    for line in lines:
        if ' = {' in line and not line.strip().startswith('#'):
            try:
                # Extract name and dict
                parts = line.split(' = ', 1)
                if len(parts) == 2:
                    name = parts[0].strip()
                    dict_str = parts[1]
                    histories[name] = ast.literal_eval(dict_str)
            except Exception as e:
                continue
    
    print(f"✅ Loaded {len(histories)} experiments from history file")
    return histories

def create_summary_from_callback(callback_manager, final_metrics=None):
    """
    Membuat summary dictionary dari callback manager dan final metrics
    
    Args:
        callback_manager: Instance YOLOCallbackManager
        final_metrics: Dictionary metrik final dari evaluasi (opsional)
        
    Returns:
        dict: Summary yang siap disimpan
    """
    history = callback_manager.history
    
    # Get final values (last epoch)
    summary = {
        'train_loss': history['train_loss'],
        'val_mAP50': history['val_mAP50'],
        'val_mAP50_95': history['val_mAP50_95'],
        'val_precision': history['val_precision'],
        'val_recall': history['val_recall'],
        'val_f1': history['val_f1'],
        'learning_rate': history['learning_rate'],
    }
    
    # Add final metrics if provided
    if final_metrics:
        summary['final_precision'] = final_metrics.get('precision', 0)
        summary['final_recall'] = final_metrics.get('recall', 0)
        summary['final_f1'] = final_metrics.get('f1_score', 0)
        summary['final_iou'] = final_metrics.get('avg_iou', 0)
    
    return summary

print("✅ History Management loaded successfully!")

## ▶️ Run Training

Jalankan cell di bawah untuk memulai training:

In [ ]:
# ==========================================
# 🚀 MAIN TRAINING FUNCTION
# ==========================================

import urllib.request
import os

def download_model_if_needed(model_path, model_url):
    """Download YOLO model from URL if not already present."""
    if not os.path.exists(model_path):
        print(f"📥 Downloading model from {model_url}...")
        urllib.request.urlretrieve(model_url, model_path)
        print(f"✅ Model downloaded to {model_path}")
    else:
        print(f"✅ Model already exists: {model_path}")

def run_training():
    """
    Main training function integrating callbacks, metrics, history saving, 
    and aggressive memory handling to prevent VS Code Window Crashes.
    """
    print("\n" + "=" * 80)
    print("🚀 STARTING YOLO TRAINING")
    print("=" * 80 + "\n")
    
    # 1. Setup output directory
    OUTPUT_DIR = DATASET_ROOT / "models" / RUNNER_NAME
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # 2. Download model if needed
    download_model_if_needed(YOLO_MODEL, YOLO_MODEL_URL)
    
    # 3. Initialize model
    print(f"🔧 Loading model: {YOLO_MODEL}")
    model = YOLO(YOLO_MODEL)
    
    # 4. Setup callbacks (EarlyStopping + Metrics Tracking + Garbage Collector)
    print(f"⚙️ Setting up callbacks...")
    callback_mgr = setup_yolo_callbacks(model, patience=PATIENCE)
    
    # 5. Clear GPU cache before training
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        import gc
        gc.collect()
        print(f"🔥 GPU: {torch.cuda.get_device_name(0)}")
        print(f"💾 VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1E9:.2f} GB\n")
    elif ENV != 'colab':
        print("⚠️ Warning: CUDA not available. Training on CPU will be highly inefficient.")
    
    # 6. Start training
    print(f"🎯 Training: {RUNNER_NAME}")
    print(f"📊 Epochs: {TARGET_EPOCHS}, Batch: {BATCH_SIZE}, Image Size: {IMG_SIZE}\n")
    
    # --- VS CODE OOM CRASH PREVENTION ---
    # VS Code renderer crashes (Code 9) when evaluating thousands of lines from tqdm progress bars over 100+ epochs.
    # Disabling tqdm and limiting verbosity forces the logging to be clean and light on memory.
    os.environ['TQDM_DISABLE'] = '1'
    os.environ['YOLO_VERBOSE'] = 'False'
    
    results = model.train(
        data=DATASET_YAML,
        project=os.path.dirname(OUTPUT_DIR),
        name=os.path.basename(OUTPUT_DIR),
        epochs=TARGET_EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        patience=PATIENCE,
        single_cls=SINGLE_CLASS,
        exist_ok=True,
        device=0 if torch.cuda.is_available() else 'cpu',
        val=True,
        
        # --- MEMORY AND PROCESS OPTIMIZATION PARAMETERS FOR 3000 IMAGES ---
        amp=True,                # Uses Automatic Mixed Precision (saves VRAM)
        workers=WORKERS,         # Reduced CPU multi-threading to prevent PyTorch RAM leaks 
        retina_masks=True,
        cache=False,             # Strictly disable caching images to RAM, rely on NVMe SSD
        verbose=False,           # Suppress large console outputs to avoid VS Code Extension Host OOM
        plots=True               # Still generate evaluation matplotlib plots directly to SSD
    )
    
    # 7. Evaluate model on test set
    print("\n" + "=" * 80)
    print("📊 EVALUATING MODEL ON TEST SET")
    print("=" * 80 + "\n")
    
    final_metrics = evaluate_model_metrics(
        model=model,
        dataset_yaml_path=DATASET_YAML,
        split='test',
        conf_threshold=0.25,
        iou_threshold=0.5
    )
    
    # 8. Create history summary from callback
    print("\n💾 Saving training history...")
    history_summary = create_summary_from_callback(callback_mgr, final_metrics)
    
    # 9. Save history to file
    save_history(history_summary, HISTORY_NAME)
    
    # 10. Final summary
    print("\n" + "=" * 80)
    print("✅ TRAINING COMPLETED SUCCESSFULLY!")
    print("=" * 80)
    print(f"📁 Model saved to:    {OUTPUT_DIR}")
    print(f"📊 History saved as:  {HISTORY_NAME}")
    print(f"📄 History file:      training_history.txt")
    print(f"\n🏆 Final Test Metrics:")
    print(f"   Precision: {final_metrics.get('precision', 0)*100:.2f}%")
    print(f"   Recall:    {final_metrics.get('recall', 0)*100:.2f}%")
    print(f"   F1-Score:  {final_metrics.get('f1_score', 0)*100:.2f}%")
    print(f"   Avg IoU:   {final_metrics.get('avg_iou', 0)*100:.2f}%")
    print("=" * 80 + "\n")
    
    # Clear GPU cache and Force Collect remaining memory variables
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        import gc
        gc.collect()
    
    return model, callback_mgr, final_metrics

# Run training
print("🔄 Ready to train! Run the cell below to start training.")

Model already exists: yolov12m-seg.pt
Starting Training: YOLOv12_Medium_Tuban
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Colab Notebooks/TA_CuttingRockDescription/datasets/C_2026_1d80_10_10_AUG/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/Colab

In [ ]:
# Jalankan training
model, callback_manager, final_metrics = run_training()

print("\n✅ Training selesai! Cek file training_history.txt untuk melihat hasil.")

## 💾 Export Model ke Folder Lokal

Simpan model hasil training ke folder `models/` di direktori notebook

In [ ]:
# ==========================================
# 💾 EXPORT MODEL KE FOLDER LOKAL
# ==========================================

import shutil
from datetime import datetime

def export_model_to_local(source_model_dir=None, export_folder="models"):
    """
    Export model weights dari hasil training ke folder lokal 'models/'
    
    Args:
        source_model_dir: Path ke folder model hasil training (default: auto dari DATASET_ROOT)
        export_folder: Nama folder tujuan (default: 'models')
    """
    # Tentukan source directory
    if source_model_dir is None:
        source_model_dir = f"{DATASET_ROOT}/models/{RUNNER_NAME}"
    
    # Buat folder export jika belum ada
    local_export_dir = os.path.join(os.getcwd(), export_folder)
    os.makedirs(local_export_dir, exist_ok=True)
    
    # Buat subfolder dengan nama experiment dan timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    export_subdir = os.path.join(local_export_dir, f"{RUNNER_NAME}_{timestamp}")
    os.makedirs(export_subdir, exist_ok=True)
    
    print("="*60)
    print("💾 EXPORTING MODEL TO LOCAL FOLDER")
    print("="*60)
    print(f"Source:      {source_model_dir}")
    print(f"Destination: {export_subdir}")
    print()
    
    # Copy weights folder
    weights_src = os.path.join(source_model_dir, "weights")
    weights_dst = os.path.join(export_subdir, "weights")
    
    if os.path.exists(weights_src):
        shutil.copytree(weights_src, weights_dst, dirs_exist_ok=True)
        print(f"✅ Copied weights folder: {weights_dst}")
        
        # List files
        weight_files = os.listdir(weights_dst)
        for wf in weight_files:
            file_size = os.path.getsize(os.path.join(weights_dst, wf)) / (1024*1024)  # MB
            print(f"   📦 {wf} ({file_size:.2f} MB)")
    else:
        print(f"⚠️ Weights folder not found: {weights_src}")
    
    # Copy important files
    files_to_copy = [
        "args.yaml",           # Training configuration
        "results.csv",         # Training metrics
        "results.png",         # Training curves
        "confusion_matrix.png",
        "confusion_matrix_normalized.png",
        "F1_curve.png",
        "P_curve.png",
        "R_curve.png",
        "PR_curve.png"
    ]
    
    print("\n📄 Copying result files...")
    for filename in files_to_copy:
        src_file = os.path.join(source_model_dir, filename)
        if os.path.exists(src_file):
            dst_file = os.path.join(export_subdir, filename)
            shutil.copy2(src_file, dst_file)
            print(f"   ✅ {filename}")
    
    # Create info file
    info_file = os.path.join(export_subdir, "model_info.txt")
    with open(info_file, 'w', encoding='utf-8') as f:
        f.write(f"Model Export Info\n")
        f.write(f"="*60 + "\n")
        f.write(f"Experiment Name: {RUNNER_NAME}\n")
        f.write(f"History Name:    {HISTORY_NAME}\n")
        f.write(f"Export Time:     {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Dataset:         {VERSION}\n")
        f.write(f"Epochs:          {TARGET_EPOCHS}\n")
        f.write(f"Batch Size:      {BATCH_SIZE}\n")
        f.write(f"Image Size:      {IMG_SIZE}\n")
        f.write(f"\nSource Path:\n{source_model_dir}\n")
        f.write(f"\nExport Path:\n{export_subdir}\n")
        f.write(f"="*60 + "\n")
    
    print(f"\n✅ Created model_info.txt")
    
    print("\n" + "="*60)
    print(f"✅ MODEL EXPORTED SUCCESSFULLY!")
    print(f"📁 Location: {export_subdir}")
    print("="*60 + "\n")
    
    return export_subdir

# Jalankan export
try:
    export_path = export_model_to_local()
    print(f"💡 Tip: Model tersimpan di folder 'models/' dan siap untuk deployment atau backup!")
except Exception as e:
    print(f"❌ Error during export: {e}")
    print(f"💡 Pastikan training sudah selesai dan model berhasil tersimpan.")

In [ ]:
def clear_cuda_cache():
    """Clear CUDA GPU memory cache."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        gc.collect()
clear_cuda_cache()
gc.collect()

## 👁️ Visualizing Model Predictions (Hollow Contours)

Visualisasi hasil prediksi model pada salah satu gambar *validation/test set*.

Titik berat visualisasi:
- **Hollow segmentation**: Hanya menampilkan garis luar (*no alpha fill*) agar persis seperti gaya anotasi CVAT.
- **Class-based Colors**: Label warna di-mapping secara terperinci untuk masing-masing kelas.

In [ ]:
import cv2
import random
import matplotlib.pyplot as plt
import numpy as np

def visualize_hollow_segmentation(model_path, images_dir):
    """
    Runs inference on a randomly selected image from images_dir and draws 
    hollow polygons (contours) for segmentation exactly like CVAT.
    """
    # Mapping Class ID to Colors (RGB format for matplotlib)
    CLASS_COLORS = {
        0: (0, 0, 255),    # Silt -> Blue
        1: (255, 165, 0),  # Sandstone -> Orange
        2: (0, 255, 0),    # Limestone -> Green
        3: (80, 80, 80),   # Coal -> Dark Gray
        4: (128, 0, 128),  # Shalestone -> Purple
        5: (0, 255, 255)   # Quartz -> Cyan
    }
    
    CLASS_NAMES = {
        0: "Silt", 1: "Sandstone", 2: "Limestone", 
        3: "Coal", 4: "Shalestone", 5: "Quartz"
    }
    
    print(f"🔍 Loading model: {model_path}")
    if not os.path.exists(model_path):
        print(f"❌ Model not found at {model_path}. Complete training first!")
        return
        
    try:
        model_inf = YOLO(model_path)
    except Exception as e:
        print(f"❌ Error loading model: {e}")
        return
        
    # Get random image from directory
    valid_exts = ('.png', '.jpg', '.jpeg')
    image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(valid_exts)]
    
    if not image_files:
        print(f"❌ No instances found in {images_dir}")
        return
        
    sample_name = random.choice(image_files)
    img_path = os.path.join(images_dir, sample_name)
    
    # Read image
    original_bgr = cv2.imread(img_path)
    original_rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)
    annotated_rgb = original_rgb.copy()
    
    # Predict (conf=0.25 is standard confidence threshold)
    print(f"👁️ Predicting on: {sample_name}")
    results = model_inf.predict(img_path, conf=0.25, verbose=False)[0]
    
    # Draw hollow segmentations
    if results.masks is not None:
        masks_xy = results.masks.xy  # Polygon coordinate arrays
        class_ids = results.boxes.cls.cpu().numpy() # Class IDs of predictions
        
        for mask, cls_id in zip(masks_xy, class_ids):
            class_id_int = int(cls_id)
            color = CLASS_COLORS.get(class_id_int, (255, 255, 255))
            
            # Format coordinates for OpenCV polylines (requires int32)
            poly_points = np.array(mask, dtype=np.int32)
            
            # Draw only the contour border (thickness=3, isClosed=True)
            if len(poly_points) > 0:
                cv2.polylines(annotated_rgb, [poly_points], isClosed=True, color=color, thickness=3)
    
    # Display comparison
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    
    axes[0].imshow(original_rgb)
    axes[0].set_title(f"Original: {sample_name}", fontsize=14)
    axes[0].axis('off')
    
    axes[1].imshow(annotated_rgb)
    axes[1].set_title("Prediction (Hollow Segmentation)", fontsize=14)
    axes[1].axis('off')
    
    # Add a legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=[c/255 for c in color], edgecolor='k', label=CLASS_NAMES.get(id, f"ID {id}")) 
                       for id, color in CLASS_COLORS.items()]
    axes[1].legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.15, 1))
    
    plt.tight_layout()
    plt.show()

# Execution:
# Best model path is generated after training. Alternatively, read it out from the folder structure.
best_model_weights = os.path.join(export_path, "weights", "best.pt") if 'export_path' in locals() else f"{DATASET_ROOT}/models/{RUNNER_NAME}/weights/best.pt"
val_images_folder = os.path.join(DATASET_ROOT, "test", "images") # Using test set to visualize unseen data

visualize_hollow_segmentation(best_model_weights, val_images_folder)